# The Development of the Auroral Substorm — Implementation
# 오로라 서브스톰의 발전 — 구현

**Paper / 논문**: Akasofu, S.-I. (1964), *Planetary and Space Science*, 12, 273–282

**목표 / Objectives**:
이 논문은 순수 현상학적 논문으로 수학적 알고리즘이 없다. 대신 Akasofu의 서브스톰 모델을 시각적으로 재현하고, 관측값을 분석한다.

This paper is purely phenomenological with no mathematical algorithms. Instead, we visually reproduce Akasofu's substorm model and analyze the observational values.

1. **Part 1**: Akasofu의 Figure 1 재현 — 극 투영도에서 서브스톰 시퀀스 시각화 / Reproduce Figure 1 — substorm sequence on polar projections
2. **Part 2**: 서브스톰 팽창 역학 시뮬레이션 — 벌지 팽창 모델링 / Substorm expansion dynamics — bulge expansion modeling
3. **Part 3**: 단일 관측소에서의 서브스톰 관측 시뮬레이션 / Single-station substorm observation simulation
4. **Part 4**: 서브스톰 특성값 비교 — Akasofu의 관측값 vs 현대 관측 / Characteristic value comparison — Akasofu vs modern observations
5. **Part 5**: 자기 교란 시뮬레이션 — substorm과 magnetic bay의 관계 / Magnetic disturbance simulation — substorm and magnetic bay relationship
6. **Part 6**: 다중 서브스톰 시퀀스 시각화 / Multiple substorm sequence visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
from matplotlib import cm

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

## Part 1: Reproducing Akasofu's Figure 1 — Polar Projection Substorm Sequence
## 파트 1: Akasofu의 Figure 1 재현 — 극 투영도 서브스톰 시퀀스

Akasofu의 가장 유명한 기여는 Figure 1의 6개 패널 극 투영 도해이다. 각 패널은 서브스톰의 한 단계를 보여주며, 지자기 극을 중심으로 오로라의 공간적 분포 변화를 나타낸다.

Akasofu's most famous contribution is the 6-panel polar projection diagram in Figure 1. Each panel shows one substorm stage, depicting the spatial distribution of aurora centered on the geomagnetic pole.

We model auroral arcs as curves in $(r, \theta)$ coordinates where:
- $r$ = colatitude from geomagnetic pole (so $r = 0°$ is the pole, $r = 30°$ is gm. lat. 60°)
- $\theta$ = magnetic local time (0° = noon, 180° = midnight)

In [ ]:
def setup_polar_panel(ax, title):
    """Set up a polar projection panel for auroral display."""
    ax.set_theta_zero_location('N')  # Noon at top
    ax.set_theta_direction(-1)  # Clockwise (east to right)
    ax.set_ylim(0, 35)  # 0 = pole, 35 = gm. lat. 55°
    ax.set_yticks([10, 20, 30])
    ax.set_yticklabels(['80°', '70°', '60°'], fontsize=7, color='gray')
    ax.set_xticks(np.radians([0, 90, 180, 270]))
    ax.set_xticklabels(['12', '18', '00', '06'], fontsize=8)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=12)
    ax.set_facecolor('k')
    ax.grid(True, alpha=0.2, color='gray')


def draw_quiet_arcs(ax, colat_base=25, n_arcs=3, mlt_range=(120, 240),
                    alpha=0.6, color='green', lw=1.5):
    """Draw quiet homogeneous arcs in the auroral zone."""
    theta = np.linspace(np.radians(mlt_range[0]), np.radians(mlt_range[1]), 100)
    for i in range(n_arcs):
        r = colat_base + i * 1.5
        ax.plot(theta, np.full_like(theta, r), color=color, alpha=alpha, lw=lw)


def draw_polar_cap_aurora(ax, alpha=0.3):
    """Draw faint polar cap bands aligned with Sun-Earth line."""
    for offset in [-2, 0, 2]:
        theta = np.linspace(np.radians(150), np.radians(210), 50)
        r = 8 + offset + 2 * np.sin(3 * theta)
        ax.plot(theta, r, color='green', alpha=alpha, lw=0.8, ls='--')


def draw_bulge(ax, colat_center=25, mlt_center=180, mlt_width=30,
               poleward_extent=5, color='lime', alpha=0.8, lw=2):
    """Draw an auroral bulge expanding poleward."""
    theta = np.linspace(np.radians(mlt_center - mlt_width),
                        np.radians(mlt_center + mlt_width), 80)
    # Gaussian poleward expansion centered at midnight
    r = colat_center - poleward_extent * np.exp(
        -0.5 * ((theta - np.radians(mlt_center)) / np.radians(mlt_width * 0.4))**2
    )
    ax.plot(theta, r, color=color, alpha=alpha, lw=lw)
    # Fill region to indicate active aurora
    ax.fill_between(theta, r, colat_center + 2,
                    color=color, alpha=0.15)


def draw_westward_surge(ax, colat=24, mlt_start=180, mlt_end=140,
                        n_folds=4, amp=2, color='yellow', lw=2):
    """Draw westward traveling surge with fold structure."""
    theta = np.linspace(np.radians(mlt_end), np.radians(mlt_start), 80)
    r = colat + amp * np.sin(n_folds * 2 * np.pi *
                              (theta - np.radians(mlt_end)) /
                              (np.radians(mlt_start) - np.radians(mlt_end)))
    # Surge amplitude decreases toward the west
    envelope = np.linspace(0.3, 1.0, 80)
    r_modulated = colat + amp * envelope * np.sin(
        n_folds * 2 * np.pi *
        (theta - np.radians(mlt_end)) /
        (np.radians(mlt_start) - np.radians(mlt_end))
    )
    ax.plot(theta, r_modulated, color=color, alpha=0.9, lw=lw)


def draw_patches(ax, n=8, colat_range=(26, 32), mlt_range=(210, 300),
                 color='cyan', alpha=0.5):
    """Draw drifting patches in the morning sector."""
    np.random.seed(42)
    for _ in range(n):
        mlt = np.random.uniform(*mlt_range)
        colat = np.random.uniform(*colat_range)
        size = np.random.uniform(0.5, 1.5)
        circle = plt.Circle((np.radians(mlt), colat), size,
                            transform=ax.transData, color=color,
                            alpha=alpha, fill=True)
        ax.add_patch(circle)


def draw_loops(ax, n=3, colat=24, mlt_start=130, spacing=8,
               color='orange', lw=1.5):
    """Draw drifting loop structures in the evening sector."""
    for i in range(n):
        mlt = mlt_start - i * spacing
        theta_loop = np.linspace(0, 2 * np.pi, 50)
        r_loop = colat + 1.5 * np.cos(theta_loop)
        t_loop = np.radians(mlt) + np.radians(2) * np.sin(theta_loop)
        ax.plot(t_loop, r_loop, color=color, alpha=0.8, lw=lw)


# Create the 6-panel substorm sequence (reproducing Akasofu's Figure 1)
fig, axes = plt.subplots(2, 3, subplot_kw={'projection': 'polar'},
                         figsize=(14, 10))
fig.suptitle("Akasofu's Auroral Substorm Sequence (1964) — Reproduced\n"
             "오로라 서브스톰 시퀀스 재현",
             fontsize=13, fontweight='bold', y=1.02)

stages = [
    'A. T=0\n(Quiet Phase)',
    'B. T=0–5 min\n(Expansive I)',
    'C. T=5–10 min\n(Expansive II)',
    'D. T=10–30 min\n(Expansive III)',
    'E. T=30 min–1 hr\n(Recovery I)',
    'F. T=1–3 hr\n(Recovery II–III)'
]

for i, (ax, title) in enumerate(zip(axes.flat, stages)):
    setup_polar_panel(ax, title)

    if i == 0:  # Quiet phase
        draw_quiet_arcs(ax, colat_base=24, n_arcs=4, mlt_range=(100, 260))
        draw_polar_cap_aurora(ax, alpha=0.3)

    elif i == 1:  # Expansive I: brightening
        draw_quiet_arcs(ax, colat_base=24, n_arcs=4, mlt_range=(100, 260),
                       alpha=0.3)
        # Brightened arc near midnight
        theta_bright = np.linspace(np.radians(150), np.radians(210), 60)
        ax.plot(theta_bright, np.full_like(theta_bright, 25),
                color='lime', lw=3, alpha=0.95)
        draw_polar_cap_aurora(ax, alpha=0.2)

    elif i == 2:  # Expansive II: bulge formation
        draw_quiet_arcs(ax, colat_base=24, n_arcs=3, mlt_range=(100, 140),
                       alpha=0.3)
        draw_quiet_arcs(ax, colat_base=24, n_arcs=3, mlt_range=(220, 260),
                       alpha=0.3)
        draw_bulge(ax, colat_center=25, poleward_extent=7, mlt_width=35)

    elif i == 3:  # Expansive III: max expansion, WTS, Ω bands
        draw_quiet_arcs(ax, colat_base=24, n_arcs=2, mlt_range=(90, 130),
                       alpha=0.4, color='lime')
        draw_bulge(ax, colat_center=25, poleward_extent=12, mlt_width=45,
                  color='lime')
        draw_westward_surge(ax, colat=24, mlt_start=175, mlt_end=120)
        # Morning sector: eastward drifting patches
        draw_patches(ax, n=6, colat_range=(25, 30), mlt_range=(230, 290))
        # Omega bands in morning
        theta_omega = np.linspace(np.radians(220), np.radians(260), 40)
        r_omega = 23 + 2 * np.abs(np.sin(4 * theta_omega))
        ax.plot(theta_omega, r_omega, color='orange', lw=2, alpha=0.8)

    elif i == 4:  # Recovery I: equatorward return, loops
        draw_quiet_arcs(ax, colat_base=26, n_arcs=2, mlt_range=(90, 130),
                       alpha=0.3)
        draw_bulge(ax, colat_center=25, poleward_extent=6, mlt_width=30,
                  color='green', alpha=0.5)
        draw_loops(ax, n=4, colat=24, mlt_start=135, spacing=7)
        draw_patches(ax, n=10, colat_range=(24, 32), mlt_range=(230, 310))
        # Equatorward arrows (conceptual)
        for mlt_deg in [160, 180, 200]:
            ax.annotate('', xy=(np.radians(mlt_deg), 24),
                       xytext=(np.radians(mlt_deg), 18),
                       arrowprops=dict(arrowstyle='->', color='white',
                                      lw=1.2, alpha=0.5))

    elif i == 5:  # Recovery II-III: return to quiet
        draw_quiet_arcs(ax, colat_base=25, n_arcs=3, mlt_range=(110, 250),
                       alpha=0.4)
        draw_patches(ax, n=5, colat_range=(27, 32), mlt_range=(250, 320),
                    alpha=0.3)
        draw_polar_cap_aurora(ax, alpha=0.15)
        # Faint returning arcs
        theta_faint = np.linspace(np.radians(140), np.radians(220), 40)
        ax.plot(theta_faint, np.full_like(theta_faint, 22),
                color='green', alpha=0.2, lw=1, ls='--')

plt.tight_layout()
plt.savefig('akasofu_fig1_reproduced.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()

## Part 2: Substorm Expansion Dynamics — Auroral Bulge Model
## 파트 2: 서브스톰 팽창 역학 — 오로라 벌지 모델

Akasofu가 기술한 벌지의 시간적 팽창을 수학적으로 모델링한다. 극 방향 이동 속도가 20–200 km/min이고, 벌지가 자정 부근에서 시작하여 동서로 팽창하는 과정을 시뮬레이션한다.

We mathematically model the temporal expansion of the bulge described by Akasofu. The poleward motion speed is 20–200 km/min, and the bulge starts near midnight and expands east-west.

**Model**: The bulge boundary is parameterized as:
$$r(\theta, t) = r_0 - v_p(t) \cdot t \cdot \exp\left(-\frac{(\theta - \theta_0)^2}{2\sigma(t)^2}\right)$$

where $r_0$ is initial colatitude, $v_p$ is poleward velocity, $\theta_0$ is midnight, and $\sigma(t)$ increases with time (east-west expansion).

In [ ]:
def bulge_model(theta, t_min, intensity='medium'):
    """Model the auroral bulge boundary as a function of MLT angle and time.

    Args:
        theta: MLT angle in radians (pi = midnight)
        t_min: time in minutes since substorm onset
        intensity: 'weak', 'medium', or 'intense'

    Returns:
        Colatitude of the bulge boundary in degrees.
    """
    # Parameters by substorm intensity (from Akasofu's observations)
    params = {
        'weak':    {'v_p': 20,  'max_colat_drop': 5,  'sigma_rate': 0.3},
        'medium':  {'v_p': 60,  'max_colat_drop': 10, 'sigma_rate': 0.5},
        'intense': {'v_p': 150, 'max_colat_drop': 18, 'sigma_rate': 0.8},
    }
    p = params[intensity]

    r0 = 25.0  # Initial colatitude (gm. lat. ~65°)
    theta0 = np.pi  # Midnight

    # Poleward expansion: velocity decreases over time (saturates)
    # 1 degree colatitude ~ 111 km
    colat_drop = min(p['v_p'] * t_min / 111.0, p['max_colat_drop'])

    # East-west expansion: sigma grows with time
    sigma = np.radians(15) + p['sigma_rate'] * np.radians(t_min)

    # Gaussian bulge shape
    r = r0 - colat_drop * np.exp(-0.5 * ((theta - theta0) / sigma)**2)

    return r


# Time evolution of bulge for three intensities
fig, axes = plt.subplots(1, 3, subplot_kw={'projection': 'polar'},
                         figsize=(15, 5.5))
fig.suptitle("Auroral Bulge Expansion: Weak vs Medium vs Intense Substorm\n"
             "오로라 벌지 팽창: 약 vs 중 vs 강 서브스톰",
             fontsize=12, fontweight='bold', y=1.05)

intensities = ['weak', 'medium', 'intense']
titles = ['Weak (v_p ~ 20 km/min)\n약한 서브스톰',
          'Medium (v_p ~ 60 km/min)\n중간 서브스톰',
          'Intense (v_p ~ 150 km/min)\n강한 서브스톰']

theta = np.linspace(np.radians(100), np.radians(260), 200)
times = [0, 2, 5, 10, 20, 30]  # minutes
colors = cm.hot(np.linspace(0.2, 0.9, len(times)))

for ax, intens, title in zip(axes, intensities, titles):
    setup_polar_panel(ax, title)

    for t, c in zip(times, colors):
        r = bulge_model(theta, t, intensity=intens)
        ax.plot(theta, r, color=c, lw=2, alpha=0.8,
                label=f'T={t} min')

    ax.legend(loc='lower left', fontsize=6, framealpha=0.5,
              labelcolor='white', facecolor='black')

plt.tight_layout()
plt.show()

# Print summary table
print("\n=== Bulge Maximum Poleward Extent (최대 극 방향 도달 위도) ===")
print(f"{'Intensity':<12} {'Max Δcolat (°)':<16} {'Max gm. lat.':<14}")
print("-" * 42)
for intens in intensities:
    r_midnight = bulge_model(np.pi, 30, intens)
    delta = 25 - r_midnight
    max_lat = 90 - r_midnight
    print(f"{intens:<12} {delta:<16.1f} {max_lat:<14.1f}°")

## Part 3: Single-Station Observation Simulation
## 파트 3: 단일 관측소 서브스톰 관측 시뮬레이션

Akasofu는 Section 3.1에서 단일 관측소가 지구 자전에 의해 15°/hr로 이동하면서 서브스톰의 각 단계를 부분적으로만 관측한다고 설명했다. 이 시뮬레이션은 다양한 MLT 위치의 관측소가 서브스톰 동안 보게 될 "오로라 활동 강도"를 모델링한다.

Akasofu explained in Section 3.1 that a single observer moves at 15°/hr with Earth's rotation, seeing only a partial view of each substorm stage. This simulation models the "auroral activity intensity" seen from stations at different MLT positions during a substorm.

In [ ]:
def substorm_activity(mlt_deg, t_min):
    """Model auroral activity intensity as a function of MLT and time.

    Based on Akasofu's phenomenological description.
    Returns relative intensity [0, 1].

    Args:
        mlt_deg: Magnetic local time in degrees (180 = midnight)
        t_min: Time in minutes since substorm onset
    """
    # Substorm phases as activity profiles
    midnight = 180.0

    # Distance from midnight in degrees
    d_midnight = abs(mlt_deg - midnight)

    # Expansive phase (0–30 min): activity peaks near midnight, spreads
    if t_min <= 0:
        return 0.05  # Quiet

    elif t_min <= 5:
        # Stage I: brightening near midnight only
        if d_midnight < 30:
            return 0.8 * np.exp(-0.5 * (d_midnight / 15)**2)
        return 0.05

    elif t_min <= 10:
        # Stage II: bulge + beginning of WTS (westward = decreasing MLT)
        evening_effect = 0.0
        if mlt_deg < midnight:  # Evening side (MLT < 180)
            wts_front = midnight - 3 * t_min  # WTS moves west
            if mlt_deg > wts_front:
                evening_effect = 0.5 * np.exp(-0.5 * ((mlt_deg - wts_front) / 20)**2)

        bulge = 0.9 * np.exp(-0.5 * (d_midnight / 25)**2)
        return max(bulge, evening_effect)

    elif t_min <= 30:
        # Stage III: max expansion, WTS + omega bands + patches
        progress = (t_min - 10) / 20  # 0 to 1

        # Bulge still active but spreading
        bulge = (0.9 - 0.3 * progress) * np.exp(-0.5 * (d_midnight / (30 + 20 * progress))**2)

        # WTS in evening
        evening_effect = 0.0
        if mlt_deg < midnight:
            wts_front = midnight - 30 - 3 * t_min
            if mlt_deg > max(wts_front, 90):
                evening_effect = 0.6 * progress * np.exp(-0.5 * ((mlt_deg - wts_front) / 15)**2)

        # Patches in morning
        morning_effect = 0.0
        if mlt_deg > midnight:
            morning_effect = 0.4 * progress * np.exp(-0.5 * ((mlt_deg - 240) / 30)**2)

        return max(bulge, evening_effect, morning_effect)

    elif t_min <= 60:
        # Recovery I: decreasing, loops in evening, patches in morning
        decay = 1.0 - (t_min - 30) / 30
        base = 0.4 * decay * np.exp(-0.5 * (d_midnight / 40)**2)

        # Loops drifting westward in evening
        loop_effect = 0.0
        if 90 < mlt_deg < 160:
            loop_effect = 0.35 * decay

        # Patches in morning
        morning = 0.0
        if mlt_deg > 200:
            morning = 0.3 * decay * np.exp(-0.5 * ((mlt_deg - 260) / 40)**2)

        return max(base, loop_effect, morning)

    elif t_min <= 120:
        # Recovery II: slow equatorward return
        decay = max(0, 1.0 - (t_min - 60) / 60)
        return 0.15 * decay + 0.05

    else:
        # Recovery III: back to quiet
        return 0.05


# Simulate what different MLT stations see during a substorm
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Panel 1: Activity vs time for fixed MLT stations
t = np.linspace(-10, 150, 500)
stations = {
    'Evening (18 MLT = 90°)': 90,
    'Pre-midnight (21 MLT = 135°)': 135,
    'Midnight (00 MLT = 180°)': 180,
    'Post-midnight (03 MLT = 225°)': 225,
    'Morning (06 MLT = 270°)': 270,
}

colors_st = ['#FF6B6B', '#FFA07A', '#FFFF00', '#7FFFD4', '#6495ED']

for (name, mlt), color in zip(stations.items(), colors_st):
    activity = [substorm_activity(mlt, ti) for ti in t]
    ax1.plot(t, activity, label=name, color=color, lw=2)

ax1.axvline(0, color='white', ls='--', alpha=0.5, lw=1)
ax1.axvline(30, color='gray', ls=':', alpha=0.5, lw=1)
ax1.text(0, 1.05, 'Onset', ha='center', fontsize=8, color='white')
ax1.text(30, 1.05, 'Recovery\nbegins', ha='center', fontsize=8, color='gray')
ax1.set_xlabel('Time since substorm onset (min) / 서브스톰 시작 후 시간 (분)')
ax1.set_ylabel('Auroral Activity Intensity / 오로라 활동 강도')
ax1.set_title('Substorm Activity at Fixed MLT Stations\n'
              '고정 MLT 관측소에서의 서브스톰 활동', fontweight='bold')
ax1.legend(loc='upper right', fontsize=8)
ax1.set_facecolor('#1a1a2e')
ax1.set_xlim(-10, 150)
ax1.set_ylim(0, 1.1)
ax1.grid(True, alpha=0.2)

# Panel 2: Activity as 2D heatmap (MLT vs time)
mlt_arr = np.linspace(60, 300, 200)
t_arr = np.linspace(-10, 150, 200)
MLT, T = np.meshgrid(mlt_arr, t_arr)
Z = np.vectorize(substorm_activity)(MLT, T)

im = ax2.pcolormesh(t_arr, mlt_arr / 15, Z, cmap='hot', shading='auto',
                    vmin=0, vmax=1)
ax2.set_xlabel('Time since substorm onset (min) / 서브스톰 시작 후 시간 (분)')
ax2.set_ylabel('Magnetic Local Time (hr) / 자기 지방시')
ax2.set_title('Substorm Activity Map (MLT vs Time)\n'
              '서브스톰 활동 지도 (MLT vs 시간)', fontweight='bold')
ax2.axhline(12, color='cyan', ls='--', alpha=0.3, lw=0.8)
ax2.set_yticks([6, 9, 12, 15, 18])
ax2.set_yticklabels(['06', '09', '12', '15', '18'])
cb = plt.colorbar(im, ax=ax2, label='Activity Intensity')

plt.tight_layout()
plt.show()

## Part 4: Akasofu's Observational Values vs Modern Observations
## 파트 4: Akasofu의 관측값 vs 현대 관측값

Akasofu (1964)가 제시한 서브스톰 특성값들을 현대 관측값과 비교한다. 60년의 시간 차이에도 불구하고, 그의 관측값이 얼마나 정확했는지 확인한다.

We compare Akasofu's (1964) substorm characteristic values with modern observations. Despite 60 years, we verify how accurate his observations were.

In [ ]:
# Comparison of Akasofu (1964) vs modern values
parameters = [
    'Substorm lifetime',
    'Expansive phase duration',
    'Recovery phase duration',
    'Poleward motion speed',
    'WTS speed',
    'Loop drift speed',
    'Patch drift speed (morning)',
    'Max poleward extent (medium)',
    'Max poleward extent (intense)',
    'Magnetic bay amplitude',
    'Quiet arc min latitude\n(moderate Dst)',
]

akasofu_vals = [
    '1–3 hr',
    '10–30 min',
    '~2 hr',
    '20–100 km/min\n(extreme: 200+)',
    '10–100 km/min\n(0.17–1.7 km/s)',
    '~30 km/min\n(~500 m/s)',
    '~20 km/min\n(~300 m/s)',
    '~75° gm. lat.',
    '80°+ gm. lat.',
    '100–2500 γ',
    '60–65° gm. lat.',
]

modern_vals = [
    '1–3 hr',
    '15–30 min\n(onset ~1–2 min)',
    '1–2 hr',
    '10–100 km/min\n(THEMIS: confirmed)',
    '5–50 km/min\n(SuperDARN)',
    '20–40 km/min',
    '15–30 km/min',
    '72–77° gm. lat.',
    '78–82° gm. lat.',
    '100–3000 nT\n(=γ)',
    '58–65° gm. lat.',
]

agreement = ['✓', '✓', '≈', '✓', '≈', '✓', '✓', '✓', '✓', '✓', '✓']

fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('off')

# Create table
table_data = list(zip(parameters, akasofu_vals, modern_vals, agreement))
col_labels = ['Parameter / 매개변수', 'Akasofu (1964)', 'Modern (2000s+)', 'Match']

table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    colWidths=[0.28, 0.25, 0.25, 0.08]
)

# Style the table
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2.2)

# Header style
for j in range(4):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Alternate row colors
for i in range(1, len(table_data) + 1):
    color = '#ecf0f1' if i % 2 == 0 else '#ffffff'
    for j in range(4):
        table[i, j].set_facecolor(color)
    # Green check marks
    if table_data[i-1][3] == '✓':
        table[i, 3].set_text_props(color='green', fontweight='bold')
    else:
        table[i, 3].set_text_props(color='orange', fontweight='bold')

ax.set_title("Akasofu (1964) vs Modern Observations — Comparison\n"
             "Akasofu (1964) vs 현대 관측 — 비교",
             fontsize=13, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("\n✓ = Excellent agreement (Akasofu's values confirmed by modern obs)")
print("≈ = Good agreement (modern values within or near Akasofu's range)")
print("\nConclusion: Akasofu's phenomenological observations from all-sky cameras")
print("were remarkably accurate, confirmed by 60 years of satellite observations.")
print("\n결론: 전천 카메라 기반 Akasofu의 현상학적 관측값은 놀라울 정도로 정확했으며,")
print("60년간의 위성 관측으로 확인되었습니다.")

## Part 5: Magnetic Disturbance Simulation — Substorm and Magnetic Bay
## 파트 5: 자기 교란 시뮬레이션 — 서브스톰과 Magnetic Bay의 관계

Akasofu는 Section 3.2에서 서브스톰 각 단계와 자기 교란의 관계를 기술했다. 저녁 섹터의 양의 임펄스, 자정 부근의 negative bay, 아침 섹터의 급속 요동을 시뮬레이션한다.

Akasofu described the relationship between substorm stages and magnetic disturbances in Section 3.2. We simulate the evening positive impulses, midnight negative bays, and morning rapid fluctuations.

The $H$-component magnetic perturbation is modeled as:
- **Evening (WTS)**: positive impulse from Hall current under the westward electrojet
- **Midnight (expansion)**: negative bay from the substorm current wedge
- **Morning (patches)**: negative bay with superposed rapid fluctuations

In [ ]:
def magnetic_bay_model(t_min, mlt_hr, intensity=500):
    """Model the H-component magnetic perturbation during a substorm.

    Args:
        t_min: Time array in minutes since onset
        mlt_hr: Magnetic local time in hours (0=midnight, 12=noon)
        intensity: Peak negative bay amplitude in nT

    Returns:
        Delta H in nT.
    """
    delta_h = np.zeros_like(t_min, dtype=float)

    for i, t in enumerate(t_min):
        if t < 0:
            delta_h[i] = 0
            continue

        # Midnight sector (22–02 MLT): strong negative bay
        if 22 <= mlt_hr or mlt_hr <= 2:
            if t <= 30:
                # Rapid decrease during expansive phase
                delta_h[i] = -intensity * (1 - np.exp(-t / 8))
            elif t <= 90:
                # Recovery: exponential decay back toward 0
                peak = -intensity * (1 - np.exp(-30 / 8))
                delta_h[i] = peak * np.exp(-(t - 30) / 40)
            else:
                peak = -intensity * (1 - np.exp(-30 / 8))
                delta_h[i] = peak * np.exp(-(t - 30) / 40)

        # Evening sector (18–22 MLT): positive impulse from WTS
        elif 18 <= mlt_hr < 22:
            surge_delay = (22 - mlt_hr) * 5  # WTS arrives later at earlier MLT
            if surge_delay < t <= surge_delay + 15:
                t_local = t - surge_delay
                delta_h[i] = 150 * np.sin(np.pi * t_local / 15)
            elif t > surge_delay + 15:
                # Possible negative component for intense surges
                t_local = t - surge_delay - 15
                delta_h[i] = -80 * np.exp(-t_local / 20)

        # Morning sector (02–06 MLT): negative bay with fluctuations
        elif 2 < mlt_hr <= 6:
            if t <= 30:
                delta_h[i] = -intensity * 0.5 * (1 - np.exp(-t / 10))
            else:
                peak = -intensity * 0.5 * (1 - np.exp(-30 / 10))
                delta_h[i] = peak * np.exp(-(t - 30) / 50)

            # Add rapid fluctuations (Pi2 pulsations, ~40-150s period)
            if 5 < t < 80:
                fluct_amp = 50 * np.exp(-abs(t - 30) / 30)
                delta_h[i] += fluct_amp * np.sin(2 * np.pi * t / 1.5)

    return delta_h


# Create synthetic magnetograms for three MLT sectors
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

t = np.linspace(-20, 150, 2000)

sectors = [
    ('Evening (20 MLT)', 20, '#FF6B6B'),
    ('Midnight (00 MLT)', 0, '#FFFF00'),
    ('Morning (04 MLT)', 4, '#6495ED'),
]

for ax, (name, mlt, color) in zip(axes, sectors):
    delta_h = magnetic_bay_model(t, mlt, intensity=600)

    ax.plot(t, delta_h, color=color, lw=1.5)
    ax.fill_between(t, delta_h, 0, alpha=0.15, color=color)
    ax.axhline(0, color='gray', ls='-', lw=0.5, alpha=0.5)
    ax.axvline(0, color='white', ls='--', lw=0.8, alpha=0.4)

    ax.set_ylabel('ΔH (nT)', fontsize=10)
    ax.set_title(name, fontsize=11, fontweight='bold', loc='left')
    ax.set_facecolor('#1a1a2e')
    ax.grid(True, alpha=0.15)

    # Annotate key features
    if mlt == 20:
        ax.annotate('Positive impulse\n(WTS passage)\n양의 임펄스 (서진서지)',
                    xy=(12, 140), fontsize=8, color='white',
                    ha='center')
    elif mlt == 0:
        ax.annotate('Negative bay\n(substorm current wedge)\n음의 bay (SCW)',
                    xy=(40, -450), fontsize=8, color='white',
                    ha='center')
    elif mlt == 4:
        ax.annotate('Rapid fluctuations\n(Pi2 pulsations)\n급속 요동',
                    xy=(30, -200), fontsize=8, color='white',
                    ha='center')

axes[-1].set_xlabel('Time since substorm onset (min) / '
                    '서브스톰 시작 후 시간 (분)')

fig.suptitle("Synthetic Magnetograms During an Auroral Substorm\n"
             "오로라 서브스톰 중 합성 자기도 (Akasofu Section 3.2 기반)",
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.show()

## Part 6: Multiple Substorm Sequence — A Disturbed Night
## 파트 6: 다중 서브스톰 시퀀스 — 교란된 밤

Akasofu는 교란 기간 중 약 12시간 동안 4개 이상의 서브스톰이 발생할 수 있으며, 새 서브스톰이 이전 서브스톰의 회복상이 끝나기 전에 시작될 수 있다고 기술했다. 자정 관측소에서의 다중 서브스톰 자기도를 시뮬레이션한다.

Akasofu described that 4+ substorms can occur in ~12 hrs during disturbed periods, with new substorms starting before the previous recovery phase completes. We simulate a multi-substorm magnetogram at a midnight station.

In [ ]:
def single_substorm_bay(t, onset, intensity, tau_rise=8, tau_decay=40):
    """Generate a single substorm negative bay waveform.

    Args:
        t: Time array in minutes
        onset: Onset time in minutes
        intensity: Peak amplitude in nT (positive value)
        tau_rise: Rise time constant (min)
        tau_decay: Decay time constant (min)

    Returns:
        Delta H contribution from this substorm.
    """
    dt = t - onset
    result = np.zeros_like(t, dtype=float)

    # Expansive phase: rapid increase of negative perturbation
    mask_exp = (dt >= 0) & (dt <= 30)
    result[mask_exp] = -intensity * (1 - np.exp(-dt[mask_exp] / tau_rise))

    # Recovery phase: exponential decay
    mask_rec = dt > 30
    peak = -intensity * (1 - np.exp(-30 / tau_rise))
    result[mask_rec] = peak * np.exp(-(dt[mask_rec] - 30) / tau_decay)

    return result


# Simulate a disturbed night with 5 substorms
np.random.seed(123)
t = np.linspace(0, 720, 5000)  # 12 hours in minutes

# Substorm parameters: (onset_time_min, intensity_nT)
substorms = [
    (30, 400),
    (150, 700),
    (270, 500),
    (380, 900),
    (520, 600),
]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8),
                                gridspec_kw={'height_ratios': [3, 1]})

# Panel 1: Composite magnetogram
total_h = np.zeros_like(t)
colors_sub = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db']

for (onset, intens), color in zip(substorms, colors_sub):
    bay = single_substorm_bay(t, onset, intens)
    # Add noise (Pi2 pulsations + instrument noise)
    noise = 20 * np.sin(2 * np.pi * t / 1.2) * np.exp(-np.abs(t - onset - 15) / 20)
    noise += np.random.normal(0, 5, len(t))
    contribution = bay + noise * (np.abs(bay) > 10)
    total_h += contribution

    # Mark onset
    ax1.axvline(onset, color=color, ls='--', alpha=0.4, lw=1)
    ax1.text(onset, 200, f'S{substorms.index((onset, intens))+1}',
             color=color, fontsize=10, fontweight='bold',
             ha='center', va='bottom')

ax1.plot(t, total_h, color='white', lw=1)
ax1.fill_between(t, total_h, 0, where=total_h < 0,
                 color='#3498db', alpha=0.2)
ax1.fill_between(t, total_h, 0, where=total_h > 0,
                 color='#e74c3c', alpha=0.2)
ax1.axhline(0, color='gray', ls='-', lw=0.5)
ax1.set_ylabel('ΔH (nT)', fontsize=11)
ax1.set_title('Synthetic Magnetogram: Disturbed Night with 5 Substorms (Midnight Station)\n'
              '합성 자기도: 5개 서브스톰이 발생한 교란된 밤 (자정 관측소)',
              fontsize=12, fontweight='bold')
ax1.set_facecolor('#0d1117')
ax1.grid(True, alpha=0.15)
ax1.set_xlim(0, 720)

# Convert x-axis to UT hours
ax1.set_xticks(np.arange(0, 721, 60))
ax1.set_xticklabels([f'{int(x/60):02d}' for x in np.arange(0, 721, 60)])

# Panel 2: Substorm activity indicator
activity = np.zeros_like(t)
for onset, intens in substorms:
    dt = t - onset
    mask = (dt >= 0) & (dt <= 30)
    activity[mask] = np.maximum(activity[mask], intens / 900)
    mask2 = (dt > 30) & (dt <= 120)
    activity[mask2] = np.maximum(activity[mask2],
                                  (intens / 900) * np.exp(-(dt[mask2] - 30) / 40))

ax2.fill_between(t, 0, activity, color='#e74c3c', alpha=0.6)
ax2.plot(t, activity, color='#e74c3c', lw=1)
ax2.set_ylabel('Activity', fontsize=10)
ax2.set_xlabel('Universal Time (hr) / 세계시', fontsize=11)
ax2.set_facecolor('#0d1117')
ax2.set_xlim(0, 720)
ax2.set_ylim(0, 1.2)
ax2.set_xticks(np.arange(0, 721, 60))
ax2.set_xticklabels([f'{int(x/60):02d}' for x in np.arange(0, 721, 60)])
ax2.grid(True, alpha=0.15)

# Add phase labels
for onset, intens in substorms:
    ax2.axvspan(onset, onset + 30, color='red', alpha=0.1)
    ax2.axvspan(onset + 30, onset + 120, color='blue', alpha=0.05)

ax2.text(360, 1.1, '■ Expansive phase  ■ Recovery phase',
         color='white', fontsize=8, ha='center',
         bbox=dict(boxstyle='round', facecolor='#0d1117', alpha=0.8))

plt.tight_layout()
plt.show()

print("\n=== Substorm Summary / 서브스톰 요약 ===")
print(f"{'#':<4} {'Onset (min)':<14} {'Onset (UT)':<12} {'Intensity (nT)':<16} {'Inter-substorm (min)'}")
print("-" * 60)
for i, (onset, intens) in enumerate(substorms):
    ut_hr = onset / 60
    gap = onset - substorms[i-1][0] if i > 0 else '—'
    print(f"S{i+1:<3} {onset:<14} {ut_hr:<12.1f} {intens:<16} {gap}")

print(f"\nNote: Akasofu stated substorms occur at intervals of 'a few hours or less'")
print(f"Our simulation: intervals of {', '.join(str(substorms[i][0] - substorms[i-1][0]) for i in range(1, len(substorms)))} min")
print(f"= {', '.join(f'{(substorms[i][0] - substorms[i-1][0])/60:.1f}' for i in range(1, len(substorms)))} hr — consistent!")

## Summary / 요약

| Part | 구현 내용 / Implementation | 핵심 결과 / Key Result |
|---|---|---|
| 1 | Akasofu Figure 1 재현 (극 투영도) | 서브스톰 6단계의 오로라 공간 분포 시각화 |
| 2 | 벌지 팽창 역학 모델 | 약/중/강 서브스톰의 극방향 도달 위도 차이 재현 |
| 3 | 단일 관측소 관측 시뮬레이션 | MLT에 따른 서브스톰 관측 특성의 차이 (Akasofu Sec. 3.1) |
| 4 | Akasofu vs 현대 관측 비교 | 60년 전 관측값이 현대 위성 관측과 놀라울 정도로 일치 |
| 5 | 자기 교란 시뮬레이션 | 저녁(양의 임펄스), 자정(negative bay), 아침(급속 요동) 재현 |
| 6 | 다중 서브스톰 시퀀스 | 교란된 밤의 반복적 서브스톰과 중첩되는 magnetic bay |

| Part | Implementation | Key Result |
|---|---|---|
| 1 | Reproduce Akasofu's Figure 1 (polar projections) | Visualize auroral spatial distribution across 6 substorm stages |
| 2 | Bulge expansion dynamics model | Reproduce poleward extent differences for weak/medium/intense substorms |
| 3 | Single-station observation simulation | MLT-dependent substorm observation characteristics (Akasofu Sec. 3.1) |
| 4 | Akasofu vs modern observation comparison | 60-year-old observations remarkably match modern satellite data |
| 5 | Magnetic disturbance simulation | Reproduce evening (positive impulse), midnight (negative bay), morning (fluctuations) |
| 6 | Multiple substorm sequence | Repetitive substorms with overlapping magnetic bays during disturbed night |

### 다음 논문과의 연결 / Connection to Next Papers

| 다음 논문 / Next Paper | 연결 / Connection |
|---|---|
| **#9 Ness (1965)** | 자기꼬리 발견 → Akasofu의 팽창상에서 방출되는 에너지가 자기꼬리에 저장됨을 확인 |
| **#10 McPherron et al. (1973)** | 성장상(growth phase)을 추가하여 3단계 모델 완성 + 위성 데이터로 검증 |
| **#11 Burton et al. (1975)** | 반복적 서브스톰에 의한 환전류 주입이 자기폭풍의 원인임을 정량화 |